In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

IMPORTING DATASETS

In [ ]:
data = 'PATH'

In [ ]:
adata = sc.read_10x_mtx(
    data,
    var_names = 'gene_symbols',
    cache = True
)
adata.var_names_make_unique()
print(adata)

SCALING \ NORMALIZING DATA

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    flavor = 'seurat_v3',
    n_top_genes = 2000
)

In [ ]:
sc.pp.normalize_total(adata, target_sum = 10000)

In [ ]:
sc.pp.log1p(adata)

In [ ]:
adata = adata[:, adata.var.highly_variable]

In [ ]:
print(adata)

In [ ]:
adata.raw = adata

In [ ]:
sc.pp.scale(adata,max_value = 10)

PERFORMING PCA

In [ ]:
sc.tl.pca(adata, svd_solver = 'arpack')

In [ ]:
print(adata)

In [ ]:
sc.pl.pca_variance_ratio(adata, log = 1)

In [ ]:
sc.pl.pca(adata)

CLUSTERING DATA BASED ON GENE EXPRESSION

In [ ]:
sc.pp.neighbors(
    adata,
    n_neighbors = 10,
    n_pcs = 15
)

In [ ]:
sc.tl.leiden(adata, resolution = 0.5)

In [ ]:
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color = ['leiden'])

FINDING DOMINANT GENE OF EACH CLUSTER

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method = 'wilcoxon')

In [ ]:
sc.pl.rank_genes_groups(adata, n_genes = 5, sharey = 0)

IDENTIFYING CELL TYPE OF CLUSTERS

In [ ]:
marker_genes = ['IL7R', 'CD8A', 'MS4A1', 'CD14', 'LYZ', 'FCGR3A', 'NKG7']
sc.pl.dotplot(adata, marker_genes, groupby='leiden')

In [ ]:
clust_annot = {
    '0': 'CD4 T-cells',
    '1': 'CD14 Monocytes',
    '2': 'CD4 T-cells',
    '3': 'B-Cells',
    '4': 'CD4 T-cells',
    '5': 'FCGR3A monocytes',
    '6': 'NK cells',
    '7': 'CD8 T-cells',
    '8': 'CD8 T-cells',
    '9': 'CD4 T-cells',
    '10': 'NA',
    '11': 'NA',
    '12': 'NA',
    '13': 'NA',
    '14': 'NA',
    '15': 'CD14 Monocytes'
}
adata.obs['cell_type'] = adata.obs['leiden'].map(clust_annot).astype('category')

In [ ]:
sc.pl.umap(adata, color = 'cell_type')

PREPARING DATA FOR SUPERVISED LEARNING

In [ ]:
adata_clean = adata[adata.obs['cell_type'] != 'NA'].copy()
adata_clean.obs['cell_type'] = adata_clean.obs['cell_type'].cat.remove_unused_categories()

In [ ]:
print(adata_clean.obs)

In [ ]:
x = pd.DataFrame(
    adata_clean.X,
    columns = adata_clean.var_names,
    index = adata_clean.obs_names
)
y = adata_clean.obs['cell_type']

In [ ]:
x

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

SUPERVISED CLASSIFIER TRAINING

In [ ]:
# Splitting Data for training and testing.

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.3, random_state = 42)

LOGISTIC REGRESSION MODEL

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(n_jobs = -1)
lr.fit(x_train, y_train)


In [ ]:
lr_pred = lr.predict(x_test)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score,confusion_matrix
print(classification_report(y_test,lr_pred))

DECISION TREE CLASSIFIER

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(max_depth = None)
dt.fit(x_train,y_train)

In [ ]:
dt_pred = dt.predict(x_test)

In [ ]:
print(classification_report(y_test, dt_pred))

In [ ]:
# Deleting unused variables
del data
del adata

In [ ]:
# Cleaning Garbage
import gc
gc.collect()

XGBOOST CLASSIFIER

In [ ]:
import xgboost as xgb
import cupy as cp
xg = xgb.XGBClassifier(
n_estimators=10000, 
random_state= 42, 
tree_method = "hist", 
device = 'cuda', 
max_depth = None, 
early_stopping_rounds = 50,
learning_rate = 0.01
)

In [ ]:
x_test_gpu = cp.array(x_test)
x_train_gpu = cp.array(x_train)
y_train_gpu = cp.array(y_train)
y_test_gpu = cp.array(y_test)
e_set = [(x_test_gpu, y_test_gpu)]

In [ ]:
xg.fit(x_train_gpu, y_train_gpu, eval_set = e_set, verbose = 0)
xg_pred = xg.predict(x_test_gpu)

In [ ]:
xg_pred1 = xg.predict(x_train_gpu)

In [ ]:
print(classification_report(y_train, xg_pred1))

In [ ]:
print(classification_report(y_test, xg_pred))

In [ ]:
print(xg.best_iteration)

RANDOM FOREST CLASSIFIER

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rfc = RandomForestClassifier(n_jobs = -1)

In [ ]:
rfc.fit(x_train, y_train)
rfc_pred = rfc.predict(x_test)

In [ ]:
print(classification_report(rfc_pred, y_test))

In [ ]:
for i in range(1,11,1):
    xgb1 = xgb.XGBClassifier(
    n_estimators=10000, 
    random_state= 42, 
    tree_method = "hist", 
    device = 'cuda', 
    max_depth = i, 
    early_stopping_rounds = 50,
    learning_rate = 0.01
    )
    xgb1.fit(x_train_gpu, y_train_gpu, eval_set = e_set, verbose = 0)
    xgb_pred = xgb1.predict(x_test_gpu)
    print("Accuracy score is of depth ",i,"is :",accuracy_score(xgb_pred,y_test))
